In [ ]:
import numpy as np

# Field Variables
receiver_diameter = 17.65  # Diameter of the receiver in meters
receiver_height = 21.6  # Height of the receiver in meters
tower_height = 200  # Height of the tower in meters
rated_power = 220  # Rated power of the system in Megawatts
time_of_day = 12  # Time of day in hours (0-23)

# Simulation Variables
min_rays = 100000
x_res = 54
y_res = 54

# CSV Variables
ideal_flux_csv = "data_dynamic/ideal_flux.csv"
dynamic_csv = "data_dynamic/heliostat_dyn_st.csv"


receiver_radius = receiver_diameter / 2  # Radius of the receiver in meters
receiver_circumference = 2 * np.pi * receiver_radius  # Circumference of the receiver in meters

In [ ]:
# Simulate
# --- Setup CoPylot ---
from copylot import CoPylot
cp = CoPylot(debug=False)
r = cp.data_create()
cp.data_set_string(r, "ambient.0.weather_file", "./climate_files/USA CA Daggett (TMY2).csv")
cp.data_set_number(r, "fluxsim.0.x_res", x_res)
cp.data_set_number(r, "fluxsim.0.y_res", y_res)
cp.data_set_number(r, "fluxsim.0.flux_hour", time_of_day)
cp.data_set_number(r, "solarfield.0.q_des", rated_power)
cp.data_set_number(r, "solarfield.0.tht", tower_height)
cp.data_set_number(r, "receiver.0.rec_diameter", receiver_diameter)
cp.data_set_number(r, "receiver.0.rec_height", receiver_height)
cp.generate_layout(r)
cp.data_set_number(r, "fluxsim.0.min_rays", min_rays)
#cp.data_set_string(r, "fluxsim.0.aim_method", "Simple aim points")
#cp.data_set_string(r, "ambient.0.sun_type", "Point sun")
cp.update_geometry(r)
cp.simulate(r)

results = cp.detail_results(r)

In [ ]:
# Extract data
import pandas as pd


def extract_heliostat_data(cp, r, results):
    """Extract heliostat positions and image sizes."""
    x_locations = np.array(results['x_location'])
    y_locations = np.array(results['y_location'])
    num_heliostats = len(x_locations)
    
    # Calculate geometric properties
    distances = np.sqrt(x_locations**2 + y_locations**2)
    alphas = np.arctan(x_locations / tower_height)  # Elevation angle to tower
    y_scale = np.sqrt(distances**2 + tower_height**2) / distances  # Scale factor for y-dimension of image size
    
    # Get image sizes for all heliostats
    image_sizes = []
    flux = []
    for i in range(num_heliostats):
        image_sizes.append(cp.get_heliostat_image_sizes(r, heliostat_id=i))
        flux.append(np.array(cp.get_heliostat_fluxmap(r, heliostat_id=i)))
    amplitudes = []
    for i in range(num_heliostats):
        amplitudes.append(flux[i].max()*1.1)
    amplitudes = np.array(amplitudes)
    print(f"Amplitudes (first 5): {flux[:5]}")
    # Scale y-dimension of image sizes
    image_sizes = np.array([(tower_height*x, tower_height*y * scale) for (x, y), scale in zip(image_sizes, y_scale)])
    x_normal = (np.arctan2(np.array(y_locations), np.array(x_locations)) - np.pi/2)*receiver_radius
    x_normal = np.mod(x_normal, receiver_circumference)
    y_normal = [receiver_height/2]*len(x_locations)
    
    return x_locations, y_locations, image_sizes, alphas, amplitudes, x_normal, y_normal

def save_all_data(image_sizes, alphas, amplitudes, x_normal, y_normal, filename=dynamic_csv):
    """Calculate and save cluster averages to CSV."""
    cluster_data = []
    
    for i in range(len(image_sizes)):

        if x_normal[i] < np.pi*receiver_diameter/2 and x_normal[i] > 0:
            xi = x_normal[i] + np.pi*receiver_diameter/2
        else:
            xi = x_normal[i] - np.pi*receiver_diameter/2

        cluster_data.append({
            'heliostat_id': i,
            'sigma_x': image_sizes[i][0],
            'sigma_y': image_sizes[i][1],
            #'alpha': alphas[i],
            #'alpha_degrees': np.degrees(alphas[i]),
            'amplitude': amplitudes[i],
            'x_locations': x_locations[i],
            'y_locations': y_locations[i],
            'x_normal': xi,
            'y_normal': y_normal[i]
        })
    
    df = pd.DataFrame(cluster_data)
    df.to_csv(filename, index=False)
    print(f"\nCluster averages saved to '{filename}'")
    
    return df

x_locations, y_locations, image_sizes, alphas, amplitudes, x_normal, y_normal = extract_heliostat_data(cp, r, results)
save_all_data(image_sizes, alphas, amplitudes, x_normal, y_normal)

assert cp.data_free(r)


In [ ]:
# %%
import numpy as np
import sys
import jax
import jax.numpy as jnp
from scipy.optimize import Bounds
import time
#import imageio.v2 as imageio # type: ignore
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
sys.path.append('./transient/')
from mmaWrapper import runMMA

jax.config.update("jax_enable_x64", False)

# ============================================================
# 1. MODEL, OPTIMIZATION, AND PLOTTING CLASS
# ============================================================

class HeliostatFluxOptimizer:
    """
    Robust heliostat aimpoint optimizer.

    Design variables
    ----------------
    a : circumferential aimpoint offset
    b : vertical aimpoint offset

    Optional robust optimization evaluates the objective over
    multiple cloud scenarios instead of only clear-sky conditions.
    """

    def __init__(
        self,
        problem_name,
        receiver_radius=receiver_radius,
        receiver_height=receiver_height,
        var_column=2,
        gamma=4.0,
        sample_heliostats=None,

        # ---------- Robust Optimization ----------
        robust=False,
        n_cloud_scenarios=20,
        variance_weight=0.05,

        cloud_radius_min=75.0,
        cloud_radius_max=250.0,

        opacity_min=0.30,
        opacity_max=0.90,

        n_clouds_range=(1, 3),
        edge_sharpness=4.0,
        cloud_combine_mode="multiplicative",

        include_clear_sky=True,

        random_seed=42
    ):

        self.problem_name = problem_name

        self.receiver_radius = receiver_radius
        self.receiver_height = receiver_height
        self.circumference = 2*np.pi*receiver_radius

        self.var_column = var_column
        self.gamma = gamma

        self.sample_heliostats = sample_heliostats

        # -----------------------------
        # Robust optimization settings
        # -----------------------------

        self.robust = robust

        self.n_cloud_scenarios = n_cloud_scenarios

        self.variance_weight = variance_weight

        self.cloud_radius_min = cloud_radius_min
        self.cloud_radius_max = cloud_radius_max

        self.opacity_min = opacity_min
        self.opacity_max = opacity_max

        self.n_clouds_range = n_clouds_range
        self.edge_sharpness = edge_sharpness
        self.cloud_combine_mode = cloud_combine_mode

        self.include_clear_sky = include_clear_sky

        self.random_seed = random_seed

        # ---------------------------------

        (
            self.F_target,
            self.heliostat_data,
            self.solution_data,
        ) = self._load_flux_map_heliostat_data(problem_name)

        self._setup_grid()

        self._setup_optical_params()

        # ---------------------------------
        # Build cloud scenarios once
        # ---------------------------------

        if self.robust:
            self.training_clouds = self.generate_cloud_scenarios(
                self.n_cloud_scenarios,
                n_clouds_range=self.n_clouds_range,
                edge_sharpness=self.edge_sharpness,
                combine_mode=self.cloud_combine_mode,
                seed=self.random_seed
            )

            self.testing_clouds = self.generate_cloud_scenarios(
                self.n_cloud_scenarios,
                n_clouds_range=self.n_clouds_range,
                edge_sharpness=self.edge_sharpness,
                combine_mode=self.cloud_combine_mode,
                seed=self.random_seed + 1000
            )

        else:

            self.training_clouds = None
            self.testing_clouds = None

        # ---------------------------------

        self.mse_and_grad = jax.jit(
            jax.value_and_grad(self.mse_loss)
        )

        self.con_and_grad = jax.jit(
            jax.value_and_grad(self.binarization_constraint)
        )

        self.oracle_mse_and_grad = jax.jit(
            jax.value_and_grad(self._mse_loss_single_cloud)
        )

        self.iter_hist = []
        self.obj_hist = []
        self.con_hist = []
        self.x_hist = []

        self.x_best = None
        self.final_mse = None
        self.cluster_abr_opt = None
        self.last_result = None

        self.F_target_max = jnp.maximum(
            jnp.max(self.F_target),
            1e-8
        )
        
    def _load_flux_map_heliostat_data(self, problem_name):
        if problem_name == "ideal_cylinder":
            csv_path = "data_dynamic/ideal_flux.csv"
            heliostat_csv_path = "data_dynamic/heliostat_dyn_st.csv"
            solution_csv_path = None
            scale_factor = 1e3
        else:
            raise ValueError(f"Unknown problem: {problem_name}")
            
        # Load target flux
        try:
            np_data = np.loadtxt(csv_path, delimiter=",")
            F_target = jnp.array(np_data)
        except:
            F_target = jnp.ones((61, 48))
        F_target = F_target / scale_factor
        
        # Load heliostat data
        # CSV columns: heliostat_id, sigma_x, sigma_y, amplitude, x_normal, y_normal
        try:
            np_data = np.loadtxt(heliostat_csv_path, delimiter=",", skiprows=1)
            heliostat_data = np_data  # Keep all 6 columns
            
            # Apply sampling if specified
            if self.sample_heliostats is not None and heliostat_data.shape[0] > self.sample_heliostats:
                idx = np.random.choice(heliostat_data.shape[0], self.sample_heliostats, replace=False)
                heliostat_data = heliostat_data[idx]
        except:
            heliostat_data = np.zeros((10, 6))
        
        # Load solution data if available
        solution_data = None
        if solution_csv_path is not None:
            try:
                np_data = np.loadtxt(solution_csv_path, delimiter=",", skiprows=1)
                solution_data = jnp.array(np_data)
            except:
                solution_data = None
            
        return F_target, heliostat_data, solution_data
    
    def _setup_grid(self):
        """Setup receiver grid (unrolled cylinder surface)"""
        self.Ny, self.Nx = self.F_target.shape[0], self.F_target.shape[1]
        self.x = jnp.linspace(0.0, self.circumference, self.Nx)
        self.y = jnp.linspace(0.0, self.receiver_height, self.Ny)
        self.X, self.Y = jnp.meshgrid(self.x, self.y)
        self.N_total = self.heliostat_data.shape[0]
    
    def _setup_optical_params(self):

        self.sigma_x = jnp.array(self.heliostat_data[:,1])

        self.sigma_y = jnp.array(self.heliostat_data[:,2])

        self.amplitude = jnp.array(self.heliostat_data[:,3])

        # Physical heliostat locations
        self.field_x = jnp.array(self.heliostat_data[:,4])
        self.field_y = jnp.array(self.heliostat_data[:,5])

        # Receiver normal locations

        self.x_normal = jnp.array(self.heliostat_data[:,6])
        self.y_normal = jnp.array(self.heliostat_data[:,7])

    # ============================================================
    # CLOUD SCENARIO GENERATION
    # ============================================================

    def generate_cloud_scenarios(
            self,
            n_scenarios,
            n_clouds_range=(1, 3),
            edge_sharpness=4.0,
            combine_mode="multiplicative",
            seed=None):
        """
        Creates a fixed collection of cloud attenuation vectors, where each
        scenario may contain multiple overlapping clouds.

        Each individual cloud uses a radial supergaussian profile:
            1 - opacity * exp(-0.5 * (r/radius)^(2*edge_sharpness))
        which gives a flat-topped disc with a sharp falloff at the edge,
        rather than a smooth/gradual Gaussian taper.

        Args:
            n_clouds_range: (min, max) number of clouds per scenario,
                inclusive. Each scenario draws a random count in this range.
            edge_sharpness: controls steepness of the cloud edge.
                1.0 = standard Gaussian falloff. Higher = sharper, more
                disc-like edge (4-6 is a reasonable "sharp cloud" range).
            combine_mode: "multiplicative" (transmission compounds under
                overlapping clouds) or "min" (darkest cloud dominates,
                no compounding).
        """

        if seed is None:
            seed = self.random_seed
        rng = np.random.default_rng(seed)

        xmin = float(np.min(self.field_x))
        xmax = float(np.max(self.field_x))

        ymin = float(np.min(self.field_y))
        ymax = float(np.max(self.field_y))

        scenarios = []

        if self.include_clear_sky:
            scenarios.append(jnp.ones(self.N_total))

        for i in range(n_scenarios):

            n_clouds = rng.integers(n_clouds_range[0], n_clouds_range[1] + 1)

            cloud_attenuations = []

            for c in range(n_clouds):

                cx = rng.uniform(xmin, xmax)
                cy = rng.uniform(ymin, ymax)

                radius = rng.uniform(
                    self.cloud_radius_min,
                    self.cloud_radius_max
                )

                opacity = rng.uniform(
                    self.opacity_min,
                    self.opacity_max
                )

                r2 = (
                    (self.field_x - cx) ** 2
                    +
                    (self.field_y - cy) ** 2
                )

                # Radial supergaussian: flat top, sharp edge falloff
                cloud_att = (
                    1.0
                    -
                    opacity
                    *
                    jnp.exp(
                        -0.5 * (r2 / radius ** 2) ** edge_sharpness
                    )
                )

                cloud_attenuations.append(cloud_att)

            cloud_attenuations = jnp.stack(cloud_attenuations)

            if combine_mode == "multiplicative":
                attenuation = jnp.prod(cloud_attenuations, axis=0)
            elif combine_mode == "min":
                attenuation = jnp.min(cloud_attenuations, axis=0)
            else:
                raise ValueError(f"Unknown combine_mode: {combine_mode}")

            scenarios.append(attenuation)

        return jnp.stack(scenarios)
    
    # ============================================================
    # CYLINDRICAL FLUX CALCULATION (from heliostat_cylinder_flux.ipynb)
    # ============================================================
    
    def _periodic_distance(self, x, x_normal):
        """
        Calculate periodic signed distance on cylinder.
        Wraps to [-circumference/2, circumference/2].
        """
        dx_signed = x - x_normal
        dx_signed = jnp.where(dx_signed > self.circumference / 2,
                              dx_signed - self.circumference,
                              jnp.where(dx_signed < -self.circumference / 2,
                                        dx_signed + self.circumference,
                                        dx_signed))
        return dx_signed
    
    def total_flux_vectorized(
        self,
        ab_norml,
        attenuation=None
    ):
        """
        Vectorized computation of total flux on cylindrical receiver.
        All heliostats are always active (rho removed as design variable).
        A heliostat contributes only within its ±90° visibility arc.
        
        Attenuation: 1/gamma only (= cos theta), matching full_2D_approx.py.
        No extra cosine incidence factor.
        
        Args:
            ab_norml: (n_heliostats, 2) array of [a_norm, b_norm]
                - a_norm: normalized X-offset [0, 1]  (0.5 = no offset)
                - b_norm: normalized Y-offset [0, 1]  (0.5 = no offset)
        
        Returns:
            flux: (Ny, Nx) total flux on receiver
        """
        n_heliostats = ab_norml.shape[0]
        
        # Denormalize offsets to physical units
        # a: X-offset range is ±90° = ±(R*π/2) for maximum physical range
        # b: Y-offset range is full height for maximum range
        a_max = self.receiver_radius * jnp.pi / 2  # ±90° arc length
        b_max = self.receiver_height / 2  # ±half height
        
        # Map [0, 1] to [-a_max, +a_max] and [-b_max, +b_max]
        a_offsets = (ab_norml[:, 0] - 0.5) * 2 * a_max  # X-offset in meters
        b_offsets = (ab_norml[:, 1] - 0.5) * 2 * b_max  # Y-offset in meters
        
        # Broadcast heliostat parameters
        x_norm = self.x_normal[:, None, None]  # (n, 1, 1)
        y_norm = self.y_normal[:, None, None]  # (n, 1, 1)
        sigma_x = self.sigma_x[:, None, None]  # (n, 1, 1)
        sigma_y = self.sigma_y[:, None, None]  # (n, 1, 1)
        # ------------------------------------------
        # Apply cloud attenuation
        # ------------------------------------------

        if attenuation is None:

            amp = self.amplitude

        else:

            amp = self.amplitude * attenuation

        amp = amp[:, None, None]
        a = a_offsets[:, None, None]  # (n, 1, 1)
        b = b_offsets[:, None, None]  # (n, 1, 1)
        
        # Broadcast grid: (n, Ny, Nx)
        X_broadcast = jnp.broadcast_to(self.X[None, :, :], (n_heliostats, self.Ny, self.Nx))
        Y_broadcast = jnp.broadcast_to(self.Y[None, :, :], (n_heliostats, self.Ny, self.Nx))
        
        # Cylindrical distances and angle
        dx = self._periodic_distance(X_broadcast, x_norm)
        dy = Y_broadcast - y_norm
        theta_diff = dx / self.receiver_radius
        
        # Reference projection mapping: projected_x = R * sin(theta)
        projected_x = self.receiver_radius * jnp.sin(theta_diff)
        
        # Gaussian arguments with offsets (a, b)
        # Key: offsets are applied in projected space
        gx = projected_x - a
        gy = dy - b
        
        # Gamma attenuation: 1/gamma = cos(theta)  [matches full_2D_approx.py g() function]
        # No extra cosine incidence factor — attenuation is purely the projection Jacobian
        gamma_factor = self.receiver_radius / jnp.sqrt(
            jnp.maximum(self.receiver_radius ** 2 - projected_x ** 2, 1e-6)
        )
        
        # Gaussian flux with 1/gamma attenuation only
        gaussian = amp * jnp.exp(
            -0.5 * ((gx / sigma_x) ** 2 + (gy / sigma_y) ** 2)
        ) / gamma_factor
        
        # Visibility masks: ±90° around x_normal; outside this range contribution is zero
        mask_x = jnp.abs(theta_diff) <= jnp.pi / 2  # ±90° visibility
        mask_y = (Y_broadcast >= 0) & (Y_broadcast <= self.receiver_height)  # Receiver bounds
        mask = mask_x & mask_y
        
        # Apply visibility mask (no rho; all heliostats contribute within their visible arc)
        flux_all = gaussian * mask
        
        # Sum over all heliostats
        flux_total = jnp.sum(flux_all, axis=0)
        
        return flux_total
    
    # ============================================================
    # LOSS AND CONSTRAINT FUNCTIONS
    # ============================================================
    
    def mse_loss(self, ab_norml_flat):

        """
        Computes either

            deterministic objective

        or

            robust cloud objective.

        """

        ab_norml = ab_norml_flat.reshape(
            (-1, self.var_column)
        )

        # ------------------------------
        # Deterministic optimization
        # ------------------------------

        if not self.robust:

            F = self.total_flux_vectorized(
                ab_norml
            )

            return jnp.mean(
                (
                    (F-self.F_target)
                    /
                    self.F_target_max
                )**2
            )

        # ------------------------------
        # Robust optimization
        # ------------------------------

        def scenario_error(attenuation):

            F = self.total_flux_vectorized(
                ab_norml,
                attenuation
            )

            return jnp.mean(
                (
                    (F-self.F_target)
                    /
                    self.F_target_max
                )**2
            )

        errors = jax.vmap(
            scenario_error
        )(self.training_clouds)

        return (
            jnp.mean(errors)
            +
            self.variance_weight*jnp.var(errors)
        )
    
    def worst_case_loss(self, ab_norml_flat):

        ab_norml = ab_norml_flat.reshape(
            (-1,self.var_column)
        )

        def scenario_error(att):

            F = self.total_flux_vectorized(
                ab_norml,
                att
            )

            return jnp.mean(
                (
                    (F-self.F_target)
                    /
                    self.F_target_max
                )**2
            )

        errors = jax.vmap(
            scenario_error
        )(self.training_clouds)

        return jnp.max(errors)
    
    def plot_cloud_scenario(self, scenario=0):

        attenuation = np.asarray(
            self.testing_clouds[scenario]
        )

        plt.figure(figsize=(7,6))

        plt.scatter(
            self.field_x,
            self.field_y,
            c=attenuation,
            cmap="viridis",
            s=10
        )

        plt.colorbar(
            label="Transmission"
        )

        plt.xlabel("Field X")

        plt.ylabel("Field Y")

        plt.title(
            f"Cloud Scenario {scenario}"
        )

        plt.axis("equal")

        plt.show()
    
    def binarization_constraint(self, ab_norml_flat):
        """Binarization constraint - DISABLED (rho removed as design variable).
        Returns a trivially satisfied value so MMA constraint interface remains intact."""
        # rho removed; constraint always satisfied
        # ab_norml = ab_norml_flat.reshape((-1, self.var_column))
        # rho = ab_norml[:, 2]
        # rho_1_rho = rho * (1.0 - rho)
        # con = jnp.mean((self.gamma * rho_1_rho) / (1 + (self.gamma - 4) * rho_1_rho)) - 1e-3
        # return con
        return jnp.array(-1e-3)  # Always satisfied (g ≤ 0)
    
    def scipy_loss(self, ab_norml_flat_np):
        """Scipy-compatible loss function"""
        ab_norml_flat_np = np.asarray(ab_norml_flat_np, dtype=np.float32)
        ab_norml_flat = jnp.array(ab_norml_flat_np, dtype=jnp.float32)
        loss, grad = self.mse_and_grad(ab_norml_flat)
        return float(loss), np.asarray(grad, dtype=np.float32).reshape((-1, 1))
    
    def scipy_con(self, ab_norml_flat_np):
        """Scipy-compatible constraint function"""
        ab_norml_flat_np = np.asarray(ab_norml_flat_np, dtype=np.float32)
        ab_norml_flat = jnp.array(ab_norml_flat_np, dtype=jnp.float32)
        con, grad = self.con_and_grad(ab_norml_flat)
        return np.array([con]).reshape((-1, 1)), np.asarray(grad, dtype=np.float32).reshape((1, -1))
    
    # ============================================================
    # INITIALIZATION AND OPTIMIZATION
    # ============================================================
    
    def initialize_clusters(self, seed=42, init_rho=0.75, init_a_range=0.2, init_b_range=0.2, NN=None):
        """
        Initialize optimization variables (a, b).
        
        Args:
            seed: random seed
            init_rho: unused (kept for API compatibility)
            init_a_range: initial range for a_norm (±init_a_range from 0.5)
            init_b_range: initial range for b_norm (±init_b_range from 0.5)
            NN: number of heliostats (default: all)
        """
        NN = self.N_total if NN is None else NN
        rng = np.random.default_rng(seed)
        
        # Note: normalized a, b are in [0, 1], where 0.5 represents no offset
        ab_np = np.zeros((NN, self.var_column))
        ab_np[:, 0] = rng.uniform(0.5 - init_a_range, 0.5 + init_a_range, NN)  # a_norm
        ab_np[:, 1] = rng.uniform(0.5 - init_b_range, 0.5 + init_b_range, NN)  # b_norm
        
        cluster_ab = jnp.array(ab_np)
        x0 = np.asarray(cluster_ab, dtype=np.float64).reshape(-1)
        
        # Bounds: a, b ∈ [0, 1]
        bounds = Bounds([0.0] * (NN * 2), [1.0] * (NN * 2))
        
        return cluster_ab, x0, bounds, rng
    
    def mma_in_fun(self, x0):
        """MMA input function (loss and constraint with gradients)"""
        f0val, df0dx = self.scipy_loss(x0)
        fval, dfdx = self.scipy_con(x0)
        return f0val, df0dx, fval, dfdx
    
    def mma_callback(self, iteration, xk, fk, gk):
        """Callback to track optimization history"""
        self.iter_hist.append(iteration)
        self.obj_hist.append(fk)
        self.con_hist.append(gk.item())
        self.x_hist.append(xk.reshape(-1).copy())

    def _mse_loss_single_cloud(self, ab_norml_flat, attenuation):
        """MSE loss for one specific attenuation vector (not averaged over clouds)."""
        ab_norml = ab_norml_flat.reshape((-1, self.var_column))
        F = self.total_flux_vectorized(ab_norml, attenuation)
        return jnp.mean(((F - self.F_target) / self.F_target_max) ** 2)
    
    def scipy_loss_oracle(self, ab_norml_flat_np, attenuation):
        x = jnp.array(np.asarray(ab_norml_flat_np, dtype=np.float32))
        loss, grad = self.oracle_mse_and_grad(x, attenuation)
        return float(loss), np.asarray(grad, dtype=np.float32).reshape((-1, 1))

    def mma_in_fun_oracle(self, x0, attenuation):
        f0val, df0dx = self.scipy_loss_oracle(x0, attenuation)
        fval, dfdx = self.scipy_con(x0)  # constraint is unchanged/trivial
        return f0val, df0dx, fval, dfdx
    
    def compute_oracle_solutions(
            self,
            clouds=None,
            warm_start=None,
            maxIterations=15,
            move_limit=0.2,
            fTolerance=1e-4,
            gTolerance=1e-2,
            minIterations=5,
            kktTol=1e-6,
            seed=42):
        """
        For each cloud scenario, run a dedicated MMA optimization to find
        the best achievable aimpoints for that specific cloud only.
        Returns the per-scenario optimal ab arrays and their MSEs.
        """
        if clouds is None:
            clouds = self.testing_clouds

        oracle_ab = []
        oracle_mse = []

        # Warm-starting from a known-decent solution (e.g. deterministic_solution)
        # converges far faster per-scenario than starting from scratch each time.
        if warm_start is not None:
            x0_base = np.asarray(warm_start, dtype=np.float64).reshape(-1)
        else:
            _, x0_base, _, _ = self.initialize_clusters(seed=seed)

        for i, attenuation in enumerate(clouds):

            x0 = x0_base.copy()

            result = runMMA(
                lambda x: self.mma_in_fun_oracle(x, attenuation),
                x0.reshape((-1, 1)),
                np.full_like(x0, 0.0).reshape((-1, 1)),
                np.full_like(x0, 1.0).reshape((-1, 1)),
                fTolerance=fTolerance,
                gTolerance=gTolerance,
                maxIterations=maxIterations,
                minIterations=minIterations,
                timeLimitSecs=3600,
                move_limit=move_limit,
                kktTol=kktTol,
                verbose=False,
                progress_callback=None,
                callback=None
            )

            ab_opt = jnp.array(np.asarray(result[0]).reshape((-1, self.var_column)))
            mse_opt = float(result[1])

            oracle_ab.append(ab_opt)
            oracle_mse.append(mse_opt)

            print(f"[oracle] scenario {i+1}/{len(clouds)}: MSE = {mse_opt:.6e}")

        self.oracle_ab = oracle_ab
        self.oracle_mse = np.asarray(oracle_mse)

        return oracle_ab, self.oracle_mse
    
    def evaluate_against_oracle(self, deterministic_ab, stochastic_ab):
        if not hasattr(self, "oracle_mse"):
            raise RuntimeError("Run compute_oracle_solutions() first.")
        
        F_det_clear = self.total_flux_vectorized(
            deterministic_ab
        )

        mse_det_clear = float(jnp.mean(
            ((F_det_clear-self.F_target)/self.F_target_max)**2
        ))

        det_errors = []
        rob_errors = []

        for attenuation in self.testing_clouds:
            F_det = self.total_flux_vectorized(deterministic_ab, attenuation)
            F_rob = self.total_flux_vectorized(stochastic_ab, attenuation)

            det_errors.append(float(jnp.mean(((F_det - self.F_target) / self.F_target_max) ** 2)))
            rob_errors.append(float(jnp.mean(((F_rob - self.F_target) / self.F_target_max) ** 2)))

        det_errors = np.asarray(det_errors)
        rob_errors = np.asarray(rob_errors)
        oracle_errors = self.oracle_mse

        det_regret = det_errors - oracle_errors
        rob_regret = rob_errors - oracle_errors

        det_degradation = 100 * (det_errors - mse_det_clear) / mse_det_clear
        rob_degradation = 100 * (rob_errors - mse_det_clear) / mse_det_clear
        oracle_degradation = 100 * (oracle_errors - mse_det_clear) / mse_det_clear

        print("\n===== REGRET VS ORACLE (per-cloud optimum) =====")
        print(f"Mean oracle MSE           : {oracle_errors.mean():.6e}")
        print(f"Mean deterministic regret : {det_regret.mean():.6e}  ({100*det_regret.mean()/oracle_errors.mean():.1f}% above oracle)")
        print(f"Mean robust regret        : {rob_regret.mean():.6e}  ({100*rob_regret.mean()/oracle_errors.mean():.1f}% above oracle)")
        print(f"Max deterministic regret  : {det_regret.max():.6e}")
        print(f"Max robust regret         : {rob_regret.max():.6e}")
        print(f"Mean deterministic degradation : {det_degradation.mean():.1f}%")
        print(f"Mean robust degradation        : {rob_degradation.mean():.1f}%")
        print(f"Mean oracle degradation         : {oracle_degradation.mean():.1f}%")

        plt.figure(figsize=(8, 5))
        plt.hist(oracle_errors, bins=20, alpha=0.6, label="Oracle (per-cloud optimal)")
        plt.hist(det_errors, bins=20, alpha=0.6, label="Deterministic")
        plt.hist(rob_errors, bins=20, alpha=0.6, label="Robust")
        plt.xlabel("Normalized MSE")
        plt.ylabel("Count")
        plt.title("Deterministic / Robust vs. Per-Cloud Oracle")
        plt.legend()
        plt.tight_layout()
        plt.show()

        return {"deterministic": det_errors, "robust": rob_errors, "oracle": oracle_errors}

    def compare_cloud_robustness(
            self,
            deterministic_ab,
            stochastic_ab,
            scenario=0):

        attenuation = self.testing_clouds[scenario]

        # Flux maps
        F_det_clear = self.total_flux_vectorized(
            deterministic_ab
        )

        F_det_cloud = self.total_flux_vectorized(
            deterministic_ab,
            attenuation
        )

        F_stoch_clear = self.total_flux_vectorized(
            stochastic_ab
        )

        F_stoch_cloud = self.total_flux_vectorized(
            stochastic_ab,
            attenuation
        )

        mse_det_clear = float(jnp.mean(
            ((F_det_clear-self.F_target)/self.F_target_max)**2
        ))

        mse_det_cloud = float(jnp.mean(
            ((F_det_cloud-self.F_target)/self.F_target_max)**2
        ))

        mse_stoch_clear = float(jnp.mean(
            ((F_stoch_clear-self.F_target)/self.F_target_max)**2
        ))

        mse_stoch_cloud = float(jnp.mean(
            ((F_stoch_cloud-self.F_target)/self.F_target_max)**2
        ))

        fig,axs = plt.subplots(
            2,
            2,
            figsize=(14,10)
        )

        plots = [
            (F_det_clear,
            f"Deterministic\nClear\nMSE={mse_det_clear:.4e}"),

            (F_det_cloud,
            f"Deterministic\nCloud\nMSE={mse_det_cloud:.4e}"),

            (F_stoch_clear,
            f"Robust\nClear\nMSE={mse_stoch_clear:.4e}"),

            (F_stoch_cloud,
            f"Robust\nCloud\nMSE={mse_stoch_cloud:.4e}")
        ]

        for ax,(F,title) in zip(axs.flat,plots):

            pcm = ax.pcolormesh(
                self.X,
                self.Y,
                F,
                shading="auto",
                cmap="viridis",
                vmin=0,
                vmax=float(jnp.max(self.F_target))
            )

            ax.set_title(title)
            ax.set_xlabel("Circumference (m)")
            ax.set_ylabel("Height (m)")

        fig.colorbar(
            pcm,
            ax=axs,
            label="Flux"
        )

        #plt.tight_layout()

        print()

        print("Performance Summary")
        print("-------------------")
        print(f"Deterministic clear : {mse_det_clear:.6e}")
        print(f"Deterministic cloud : {mse_det_cloud:.6e}")
        print(f"Robust clear        : {mse_stoch_clear:.6e}")
        print(f"Robust cloud        : {mse_stoch_cloud:.6e}")

        print()

        print(
            f"Cloud degradation (deterministic): "
            f"{100*(mse_det_cloud-mse_det_clear)/mse_det_clear:.1f}%"
        )

        print(
            f"Cloud degradation (robust): "
            f"{100*(mse_stoch_cloud-mse_stoch_clear)/mse_stoch_clear:.1f}%"
        )
    
    def evaluate_robustness(
            self,
            deterministic_ab,
            stochastic_ab,
            plot_worst=True,
            optimize_cloud=False
            ):

        """
        Compare deterministic and stochastic aimpoints on
        previously unseen cloud scenarios.

        Parameters
        ----------
        deterministic_ab : ndarray
            Aimpoints from deterministic optimization.

        stochastic_ab : ndarray
            Aimpoints from stochastic optimization.

        plot_worst : bool
            Plot the worst test cloud if True.
        """

        if optimize_cloud:

           print("In progress...")

        det_errors_cloud = []
        rob_errors_cloud = []

        det_errors_clear = []
        rob_errors_clear = []

        det_fluxes_cloud = []
        rob_fluxes_cloud = []

        det_fluxes_clear = []
        rob_fluxes_clear = []

        det_degrad = []
        rob_degrad = []

        F_det_clear = self.total_flux_vectorized(
            deterministic_ab,
        )

        F_rob_clear = self.total_flux_vectorized(
            stochastic_ab,
        )

        det_clear_mse = jnp.mean(
                ((F_det_clear-self.F_target)/self.F_target_max)**2
            )
        
        rob_clear_mse = jnp.mean(
                ((F_rob_clear-self.F_target)/self.F_target_max)**2
            )

        # ------------------------------------
        # Evaluate every unseen cloud
        # ------------------------------------

        for attenuation in self.testing_clouds:

            F_det_cloud = self.total_flux_vectorized(
                deterministic_ab,
                attenuation
            )

            F_rob_cloud = self.total_flux_vectorized(
                stochastic_ab,
                attenuation
            )

            det_fluxes_cloud.append(F_det_cloud)
            rob_fluxes_cloud.append(F_rob_cloud)
            det_fluxes_clear.append(F_det_clear)
            rob_fluxes_clear.append(F_rob_clear)

            det_cloud_mse = jnp.mean(
                    ((F_det_cloud-self.F_target)/self.F_target_max)**2
                )
            
            rob_cloud_mse = jnp.mean(
                    ((F_rob_cloud-self.F_target)/self.F_target_max)**2
                )
            

            det_errors_cloud.append(det_cloud_mse)
            rob_errors_cloud.append(rob_cloud_mse)
            det_errors_clear.append(det_clear_mse)
            rob_errors_clear.append(rob_clear_mse)

            det_degrad.append(
                100 * (det_cloud_mse - det_clear_mse) / det_clear_mse
            )

            rob_degrad.append(
                100 * (rob_cloud_mse - rob_clear_mse) / rob_clear_mse
            )

        det_errors = np.asarray(det_errors_cloud)
        rob_errors = np.asarray(rob_errors_cloud)
        det_errors_clear = np.asarray(det_errors_clear)
        rob_errors_clear = np.asarray(rob_errors_clear)
        det_degrad = np.asarray(det_degrad)
        rob_degrad = np.asarray(rob_degrad)

        # ------------------------------------
        # Statistics
        # ------------------------------------

        print("\n========== ROBUSTNESS RESULTS ==========")

        print(f"Testing clouds          : {len(det_errors)}")

        print()

        print(f"Mean MSE (Det)          : {det_errors.mean():.6e}")
        print(f"Mean MSE (Robust)       : {rob_errors.mean():.6e}")

        print()

        print(f"Median MSE (Det)        : {np.median(det_errors):.6e}")
        print(f"Median MSE (Robust)     : {np.median(rob_errors):.6e}")

        print()

        print(f"Std Dev (Det)           : {det_errors.std():.6e}")
        print(f"Std Dev (Robust)        : {rob_errors.std():.6e}")

        print()

        print(f"Worst Case (Det)        : {det_errors.max():.6e}")
        print(f"Worst Case (Robust)     : {rob_errors.max():.6e}")

        print()

        print(f"95th Percentile (Det)   : {np.percentile(det_errors,95):.6e}")
        print(f"95th Percentile (Robust): {np.percentile(rob_errors,95):.6e}")

        print(f"Mean Cloud Degradation (Det) : {det_degrad.mean():.2f}%")
        print(f"Mean Cloud Degradation (Robust) : {rob_degrad.mean():.2f}%")

        print()

        mean_gain = (
            (det_errors.mean()-rob_errors.mean())
            / det_errors.mean()
        )*100

        worst_gain = (
            (det_errors.max()-rob_errors.max())
            / det_errors.max()
        )*100

        print(f"Mean Improvement        : {mean_gain:.2f}%")
        print(f"Worst Improvement       : {worst_gain:.2f}%")

        # ------------------------------------
        # Histograms
        # ------------------------------------

        plt.figure(figsize=(8,5))

        plt.hist(
            det_errors,
            bins=20,
            alpha=0.6,
            label="Deterministic"
        )

        plt.hist(
            rob_errors,
            bins=20,
            alpha=0.6,
            label="Robust"
        )

        plt.xlabel("Normalized MSE")
        plt.ylabel("Count")

        plt.title("Performance on Unseen Cloud Scenarios")

        plt.legend()

        plt.tight_layout()

        plt.show()

        # ------------------------------------
        # Plot worst deterministic cloud
        # ------------------------------------

        if plot_worst:

            idx = np.argmax(det_errors)

            attenuation = self.testing_clouds[idx]

            fig,axs = plt.subplots(
                1,
                2,
                figsize=(12,5)
            )

            pcm = axs[0].pcolormesh(
                self.X,
                self.Y,
                det_fluxes_cloud[idx],
                shading="auto"
            )

            axs[0].set_title(
                f"Deterministic\nWorst Cloud\nMSE={det_errors[idx]:.3e}"
            )

            axs[1].pcolormesh(
                self.X,
                self.Y,
                rob_fluxes_cloud[idx],
                shading="auto"
            )

            axs[1].set_title(
                f"Robust\nSame Cloud\nMSE={rob_errors[idx]:.3e}"
            )

            fig.colorbar(
                pcm,
                ax=axs
            )

            plt.tight_layout()

            plt.show()

        return {
            "deterministic": det_errors,
            "robust": rob_errors
        }

    def optimize_mma(self, seed=42, init_rho=0.75, n_st=2, fTolerance=1e-4, gTolerance=1e-2, 
                    maxIterations=100, minIterations=10, timeLimitSecs=3600, move_limit=0.2, 
                    kktTol=1e-6, verbose=False, getGIF=False):
        """
        Run MMA optimization to find optimal (a, b) for each heliostat.
        
        Args:
            seed: random seed
            init_rho: unused (kept for API compatibility; rho removed as design variable)
            n_st: number of random restarts
            Other args: MMA optimizer parameters
        """
        cluster_ab, x0, bounds, rng = self.initialize_clusters(seed=seed, init_rho=init_rho)
        
        # Benchmark function evaluation time
        nn = 5
        st = time.perf_counter()
        for _ in range(nn):
            self.mma_in_fun(x0)
        et = time.perf_counter()
        print(f"Avg time taken for one fun eval: {(et - st)/nn:.6f} seconds")
        
        fbest = np.inf
        for i in range(n_st):
            self.iter_hist = []
            self.obj_hist = []
            self.con_hist = []
            self.x_hist = []
            
            result = runMMA(
                self.mma_in_fun,
                x0.reshape((-1, 1)),
                np.full_like(x0, 0.0).reshape((-1, 1)),  # a, b lower bound = 0
                np.full_like(x0, 1.0).reshape((-1, 1)),  # a, b upper bound = 1
                fTolerance=fTolerance,
                gTolerance=gTolerance,
                maxIterations=maxIterations,
                minIterations=minIterations,
                timeLimitSecs=timeLimitSecs,
                move_limit=move_limit,
                kktTol=kktTol,
                verbose=verbose,
                progress_callback=None,
                callback=self.mma_callback
            )
            
            if result[1] < fbest:
                self.x_best = np.asarray(result[0]).reshape(-1)
                fbest = result[1]
                x0 = self.x_best.copy()
            else:
                # Random restart: uniformly sample a, b in [0, 1]
                x0 = rng.uniform(0, 1, x0.size)
            
            print(f"Restart {i+1}/{n_st}, Best Objective so far: {float(fbest):.6e}, " + 
                  f"solved by MMA in {float(result[-1]):.2f} seconds, took {int(result[-2])} iterations")
            self.last_result = result
        
        self.cluster_abr_opt = jnp.array(self.x_best.reshape((-1, self.var_column)))
        self.final_mse, _ = self.scipy_loss(self.x_best)
        
        if getGIF:
            self.create_gif(gif_name="mma_flux_optimization_evolution.gif", fps=5, hold_last=20)
        
        return self.x_best, self.final_mse, self.cluster_abr_opt
    
    # ============================================================
    # PLOTTING AND VISUALIZATION
    # ============================================================
    
    def plot_target(self):
        """Plot target flux distribution"""
        plt.figure(figsize=(10, 8))
        plt.pcolormesh(self.X, self.Y, self.F_target, shading='auto', cmap='viridis')
        plt.colorbar(label='Flux [kW/m²]')
        plt.xlabel('Circumference [m]')
        plt.ylabel('Height [m]')
        plt.title(f'Target Receiver Flux Map\nType: {self.problem_name}')
        plt.axis('equal')
        plt.tight_layout()
        plt.show()
    
    def plot_flux_map(self, F, title='Flux Map'):
        """Plot a flux distribution"""
        plt.figure(figsize=(10, 8))
        plt.pcolormesh(self.X, self.Y, F, shading='auto', cmap='viridis')
        plt.colorbar(label='Flux [kW/m²]')
        plt.xlabel('Circumference [m]')
        plt.ylabel('Height [m]')
        plt.title(title)
        plt.axis('equal')
        plt.tight_layout()
        plt.show()
    
    def plot_history(self):
        """Plot optimization history"""
        if not self.iter_hist:
            print("No history to plot.")
            return
        fig, ax1 = plt.subplots()
        ax2 = ax1.twinx()
        ax1.set_xlabel("Iteration", fontweight='bold')
        ax1.set_ylabel("Normalized objective function", color="b", fontweight='bold')
        ax2.set_ylabel("Constraint value", color="r", fontweight='bold')
        ax2.yaxis.set_label_position("right")
        ax2.yaxis.tick_right()
        ax1.plot(self.iter_hist, self.obj_hist, 'b-o', label="Objective")
        ax2.plot(self.iter_hist, self.con_hist, 'r-o', label="Constraint")
        ax1.tick_params(axis='y', labelcolor='b')
        ax2.tick_params(axis='y', labelcolor='r')
        plt.pause(0.01)
    
    def create_gif(self, gif_name="flux_opt_evolution.gif", fps=5, hold_last=20):
        """Create animated GIF of optimization evolution"""
        if not self.x_hist:
            print("No history to create GIF.")
            return
        frames = []
        vmin = float(jnp.min(self.F_target))
        vmax = float(jnp.max(self.F_target))
        for idx, xk in enumerate(self.x_hist):
            cluster_ab_iter = jnp.array(xk.reshape((-1, self.var_column)))
            F_iter = self.total_flux_vectorized(cluster_ab_iter)
            fig, ax = plt.subplots(figsize=(8, 6), dpi=150)
            ax.pcolormesh(self.X, self.Y, F_iter, shading='auto', cmap='viridis', vmin=vmin, vmax=vmax)
            ax.set_xlabel('Circumference [m]')
            ax.set_ylabel('Height [m]')
            ax.set_title(f'Flux Optimization Evolution, Iteration {idx+1}')
            ax.set_aspect('equal')
            fig.canvas.draw()
            frame = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
            frame = frame.reshape(fig.canvas.get_width_height()[::-1] + (3,))
            frames.append(frame)
            plt.close(fig)
        if hold_last > 0:
            for _ in range(hold_last):
                frames.append(frames[-1])
        imageio.mimsave(gif_name, frames, fps=fps, loop=0)
        print(f"GIF saved as {gif_name}")
    
    def plot_optimized_vs_target(self, show_heliostat_positions=True):
        """Plot optimized flux vs target flux side by side"""
        if self.cluster_abr_opt is None:
            print("Run optimize_mma first.")
            return
        
        #print(f"Final Objective: {self.final_mse[0]:.4e}")
        F_opt = self.total_flux_vectorized(self.cluster_abr_opt)
        
        # Denormalize offsets
        a_max = self.receiver_radius * jnp.pi / 2
        b_max = self.receiver_height / 2
        a_offsets = (self.cluster_abr_opt[:, 0] - 0.5) * 2 * a_max
        b_offsets = (self.cluster_abr_opt[:, 1] - 0.5) * 2 * b_max
        
        # Active heliostats: those whose aimed point lies within ±90° of their x_normal.
        # Outside this arc the ±90° visibility mask zeroes out their contribution.
        active_mask = jnp.abs(a_offsets) <= a_max
        active_heliostats = jnp.sum(active_mask)
        print(f"Total number of active heliostats: {int(active_heliostats)} out of total {self.N_total} heliostats")
        
        # Calculate actual heliostat aim positions (x_normal + a, y_normal + b)
        x_positions = (self.x_normal + a_offsets) % self.circumference
        y_positions = self.y_normal + b_offsets
        
        x_active = x_positions[active_mask]
        y_active = y_positions[active_mask]
        
        # Plot
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        
        # Target flux
        im0 = axes[0].pcolormesh(self.X, self.Y, self.F_target, shading='auto', cmap='viridis')
        axes[0].set_xlabel('Circumference [m]')
        axes[0].set_ylabel('Height [m]')
        axes[0].set_title('Target Flux Map (F_target)')
        # axes[0].axis('equal')
        plt.colorbar(im0, ax=axes[0], label='Flux [KW/m^2]')
        
        # Optimized flux
        im1 = axes[1].pcolormesh(self.X, self.Y, F_opt, shading='auto', cmap='viridis')
        if show_heliostat_positions:
            axes[1].plot(x_active, y_active, 'rx', markersize=4, label='Optimized Centers', alpha=0.7)
        axes[1].set_xlabel('Circumference [m]')
        axes[1].set_ylabel('Height [m]')
        axes[1].set_title('Optimized Flux Map (F)')
        # axes[1].axis('equal')
        plt.colorbar(im1, ax=axes[1], label='Flux [KW/m^2]')
        axes[1].legend()
        
        # Add informative suptitle with MSE
        # fig.suptitle(
        #     f'Heliostat Flux Optimization with {self.N_total} heliostats with \n'
        #     f'active heliostats = {int(active_heliostats)} \n'
        #     r'$\mathrm{mean}\left(\!\left(\frac{F - F_{target}}{\max(F_{target})}\right)^{2}\right)$'
        #     f' = {np.round(float(self.final_mse[0]), decimals=4)}',
        #     fontsize=16,
        #     fontweight='bold'
        # )
        plt.tight_layout()
        plt.show()

    def plot_3d_surface(self):
        """Plot target and optimized flux as 3D surfaces."""
        if self.cluster_abr_opt is None:
            print("Run optimize_mma first.")
            return

        F_opt = self.total_flux_vectorized(self.cluster_abr_opt)

        fig = plt.figure(figsize=(16, 8))
        ax = fig.add_subplot(111, projection='3d')
        surf1 = ax.plot_surface(self.X, self.Y, F_opt, cmap='viridis', alpha=0.5, linewidth=0, antialiased=True)
        surf2 = ax.plot_surface(self.X, self.Y, self.F_target, cmap='seismic', alpha=0.5, linewidth=0, antialiased=True)
        ax.set_xlabel('X [m]')
        ax.set_ylabel('Y [m]')
        ax.set_zlabel('Flux [kW/m²]')
        ax.set_title(f'Heliostat Flux Optimization with {self.N_total} heliostats', fontsize=14, fontweight='bold')
        fig.colorbar(surf2, ax=ax, shrink=0.5, aspect=15, label='Target Flux')
        fig.colorbar(surf1, ax=ax, shrink=0.5, aspect=15, label='Optimized Flux')
        plt.tight_layout()
        plt.show()


    def plot_y_slices(self, n_slices=6):
        """Plot Y-direction slices comparing target and optimized flux."""
        if self.cluster_abr_opt is None:
            print("Run optimize_mma first.")
            return

        F_opt = self.total_flux_vectorized(self.cluster_abr_opt)
        y_indices = np.linspace(0, self.Y.shape[0] - 1, n_slices, dtype=int)
        fig, axes = plt.subplots(n_slices, 1, figsize=(7, 2 * n_slices), sharex=True)
        if n_slices == 1:
            axes = [axes]
        for ax, i in zip(axes, y_indices):
            ax.plot(self.X[i, :], F_opt[i, :], label='Optimized Flux', linewidth=2)
            ax.plot(self.X[i, :], self.F_target[i, :], '--', label='Target Flux', linewidth=2, color='red')
            ax.set_ylabel('Flux [kW/m²]')
            ax.set_title(f'Y = {self.Y[i, 0]:.2f} m')
            ax.grid(True)
        axes[-1].set_xlabel('X [m]')
        axes[0].legend(loc='best')
        plt.tight_layout()
        plt.show()

    def plot_cloud_comparison(
        self,
        cluster_ab,
        scenario=0):

        F_clear = self.total_flux_vectorized(
            cluster_ab
        )

        attenuation = self.testing_clouds[scenario]

        F_cloud = self.total_flux_vectorized(
            cluster_ab,
            attenuation
        )

        fig,axs = plt.subplots(
            1,
            3,
            figsize=(18,6)
        )

        im=axs[0].pcolormesh(
            self.X,
            self.Y,
            self.F_target,
            shading="auto"
        )

        axs[0].set_title("Target")

        axs[1].pcolormesh(
            self.X,
            self.Y,
            F_clear,
            shading="auto"
        )

        axs[1].set_title("Clear Sky")

        axs[2].pcolormesh(
            self.X,
            self.Y,
            F_cloud,
            shading="auto"
        )

        axs[2].set_title("Cloud Scenario")

        plt.colorbar(im,ax=axs)

        #plt.tight_layout()

        plt.show()


In [ ]:
# Initialize cylindrical heliostat flux optimizer
# Design variables: (a, b) for each heliostat (all normalized to [0, 1])
# - a: X-offset from x_normal, [0,1] maps to [-90°, +90°] arc length
# - b: Y-offset from y_normal, [0,1] maps to [-half_height, +half_height]

problem_name = "ideal_cylinder"
receiver_radius = receiver_radius
receiver_height = receiver_height

model = HeliostatFluxOptimizer(

    problem_name="ideal_cylinder",

    receiver_radius=receiver_radius,

    receiver_height=receiver_height,

    var_column=2,

    gamma=4,

    sample_heliostats=None,

    robust=False,

    random_seed=42
)
# Initial guess check before MMA optimization
# a=0.5, b=0.5 → no offset for all heliostats (aimed at x_normal, y_normal)

ab_guess = np.zeros((model.N_total, 2), dtype=np.float32)
ab_guess[:, 0] = 0.5  # a = 0.5 → zero x-offset (at x_normal)
ab_guess[:, 1] = 0.5  # b = 0.5 → zero y-offset (at y_normal)

F_guess = model.total_flux_vectorized(jnp.array(ab_guess))
mse_guess = jnp.mean(((F_guess - model.F_target) / model.F_target_max) ** 2)

print("Initial guess (before MMA): a={:.1f}, b={:.1f} for all heliostats".format(ab_guess[0, 0], ab_guess[0, 1]))
print(f"Initial normalized MSE: {float(mse_guess):.6e}")

model.plot_flux_map(F_guess, title='Initial Guess Flux Map (a=0.5, b=0.5)')

# ============================================================
# 3. OPTIMIZATION (MMA)
# ============================================================
x_best, final_mse, cluster_abr_opt = model.optimize_mma(
    seed=42,
    init_rho=0.75,
    n_st=1,
    fTolerance=1e-4,
    gTolerance=1e-2,
    maxIterations=15,
    minIterations=5,
    timeLimitSecs=3600,
    move_limit=0.2,
    kktTol=1e-6,
    verbose=False,
    getGIF=False
)
model.plot_history()

deterministic_solution = model.cluster_abr_opt

In [ ]:
# Initialize cylindrical heliostat flux optimizer
# Design variables: (a, b) for each heliostat (all normalized to [0, 1])
# - a: X-offset from x_normal, [0,1] maps to [-90°, +90°] arc length
# - b: Y-offset from y_normal, [0,1] maps to [-half_height, +half_height]

problem_name = "ideal_cylinder"
receiver_radius = receiver_radius
receiver_height = receiver_height

model = HeliostatFluxOptimizer(

    problem_name="ideal_cylinder",

    receiver_radius=receiver_radius,

    receiver_height=receiver_height,

    var_column=2,

    gamma=4,

    sample_heliostats=None,

    robust=True,

    n_clouds_range=(5, 15),

    n_cloud_scenarios=10,

    variance_weight=0.10,

    cloud_radius_min=100,

    cloud_radius_max=800,

    opacity_min=0.35,

    opacity_max=0.60,

    include_clear_sky=True,

    random_seed=42
)

ab_guess = np.zeros((model.N_total, 2), dtype=np.float32)
ab_guess[:, 0] = 0.5  # a = 0.5 → zero x-offset (at x_normal)
ab_guess[:, 1] = 0.5  # b = 0.5 → zero y-offset (at y_normal)

F_guess = model.total_flux_vectorized(jnp.array(ab_guess))
mse_guess = jnp.mean(((F_guess - model.F_target) / model.F_target_max) ** 2)

print("Initial guess (before MMA): a={:.1f}, b={:.1f} for all heliostats".format(ab_guess[0, 0], ab_guess[0, 1]))
print(f"Initial normalized MSE: {float(mse_guess):.6e}")

model.plot_flux_map(F_guess, title='Initial Guess Flux Map (a=0.5, b=0.5)')

# ============================================================
# 3. OPTIMIZATION (MMA)
# ============================================================
x_best, final_mse, cluster_abr_opt = model.optimize_mma(
    seed=42,
    init_rho=0.75,
    n_st=1,
    fTolerance=1e-4,
    gTolerance=1e-2,
    maxIterations=15,
    minIterations=5,
    timeLimitSecs=3600,
    move_limit=0.2,
    kktTol=1e-6,
    verbose=False,
    getGIF=False
)
model.plot_history()

stochastic_solution = model.cluster_abr_opt

print(
    f"Tests clouds: {len(model.testing_clouds)}"
    f"Training clouds: {len(model.training_clouds)}"
)

for i in range(5):

    model.plot_cloud_scenario(i+1)


In [ ]:
# Best Case Calculation
model.compute_oracle_solutions(
    clouds=model.testing_clouds,
    warm_start=deterministic_solution.reshape(-1),
    maxIterations=15
)


In [ ]:
model.evaluate_against_oracle(
    deterministic_ab=deterministic_solution,
    stochastic_ab=stochastic_solution
)

In [ ]:
model.evaluate_robustness(
    deterministic_ab=deterministic_solution,
    stochastic_ab=stochastic_solution,
    plot_worst=False
)

In [ ]:
model.compare_cloud_robustness(
    deterministic_solution,
    stochastic_solution,
    scenario=8
)